# Train Roman's LoRA on Kaggle (Z-Image Turbo, free)

Runs unattended with **Save Version → Save & Run All**: no clicking between cells.

- **Input:** the private Kaggle dataset made from `outputs/lora/dataset_<date>.zip`
  (`uv run ca lora prepare`): images + `.txt` captions that start with the trigger word.
  To continue an earlier run, also attach that version's output (Add Input → Notebook Output).
- **Settings:** Accelerator **GPU T4 x2**, Internet **on**.
- **Trainer:** Ostris AI Toolkit with its example defaults (rank 16, lr 1e-4) and its
  low-VRAM options (float8 transformer and text encoder, low_vram, cached
  latents and text embeddings), with Ostris' Turbo training adapter.
- **Precision:** never fp16 here. Z-Image overflows fp16 (the loss turns NaN, AI Toolkit then
  logs it as 0 and nothing is learned). bf16 is used when the GPU passes a short bf16 test
  (the T4 only emulates it: correct but slower), else fp32.
- **Guard:** if the loss is 0, NaN or infinite for the first 30 steps, the run stops at once.
- **T4 settings:** one GPU, 1200 steps, buckets up to 768 px, PyTorch AdamW (no bitsandbytes),
  no baseline samples; free RAM and GPU memory are logged every 20 steps, and a crash prints
  its Python stack (faulthandler) into `train_log.txt`.
- **Output (`/kaggle/working/lora_output/<job>/`):** a checkpoint and sample images every 100
  steps (the last 4 checkpoints are kept), `train_log.txt`, `status.json`, and all of it in
  one `<job>_outputs.zip`.
- **Resume:** the newest usable checkpoint (in `/kaggle/working` or an attached earlier output)
  is picked up and training continues from it; one that learned nothing is set aside.
- **Time:** training stops after `TIME_BUDGET_HOURS`, well inside the session limit.

The photos stay in `/tmp` and are not part of the output.

In [ ]:
# Settings: AI Toolkit's example defaults, adjusted for a free T4 (no other tuning).
STEPS = 1200
RANK = 16  # linear and linear_alpha
LEARNING_RATE = 1e-4
SAVE_EVERY = 100  # a checkpoint + sample images every N steps
KEEP_CHECKPOINTS = 4  # only the last 4, to save disk
TIME_BUDGET_HOURS = 9.0  # training stops here; the session limit is about 12 h
GUARD_STEPS = 30  # stop if the loss is 0 / NaN / inf for this many first steps
MIN_IMAGES = 9
RESOLUTIONS = [512, 768]  # buckets up to 768 px (1024 took ~45 s/step on the T4)
FALLBACK_RESOLUTIONS = [512]  # used once if the GPU runs out of memory early
SAMPLE_SIZE = 768  # sample images (1024 took ~4 min each on the T4)
OPTIMIZER = "adamw"  # PyTorch AdamW: no bitsandbytes native code in the training loop
MEMORY_LOG_EVERY = 20  # free RAM and GPU memory into train_log.txt every N steps
BASE_MODEL = "Tongyi-MAI/Z-Image-Turbo"
ADAPTER_REPO = "ostris/zimage_turbo_training_adapter"
TOOLKIT_REPO = "https://github.com/ostris/ai-toolkit.git"
SKIP_PACKAGES = {"torch", "torchvision", "torchaudio"}  # Kaggle's CUDA build stays
# Exact versions to install instead of a relaxed pin (status.json "relaxed_pins"), e.g.
# {"optimum-quanto": "optimum-quanto==0.2.7"} when the log implicates that package.
PINNED: dict[str, str] = {
    "scipy": "scipy==1.14.1",  # AI Toolkit pins 1.12.0 (no Python 3.13 wheel); 1.14.1 has one
}
WORK = "/kaggle/working"
OUTPUT_DIR = f"{WORK}/lora_output"  # checkpoints and samples: saved as notebook output
SCRATCH = "/tmp/ca_lora"  # dataset copy, toolkit and base models: not saved
SAMPLE_PROMPTS = [
    "[trigger], a man in a navy suit in a modern office, photorealistic portrait, window light",
    "[trigger], a man smiling at a cafe table with a laptop, natural light, photo",
    "[trigger], close-up portrait of a man, neutral grey background, studio light",
    "[trigger], a man presenting in front of a screen in a meeting room, photo",
    "a man in a navy suit in a modern office, photorealistic portrait, window light",
]

In [ ]:
import json
import os
import subprocess
import time
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU only (T4 x2 has two)
import torch  # noqa: E402 - after CUDA_VISIBLE_DEVICES


def sh(command: str) -> None:
    """Run a shell command and stop the notebook if it fails (only used before training)."""
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True)


if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2, then Save & Run All again.")
MAJOR, MINOR = torch.cuda.get_device_capability(0)


def bf16_works() -> bool:
    """bf16 matmul, conv and softmax on this GPU give finite results close to fp32."""
    try:
        torch.manual_seed(0)
        a = torch.randn(512, 512, device="cuda")
        image = torch.randn(1, 16, 64, 64, device="cuda")
        kernel = torch.randn(32, 16, 3, 3, device="cuda")
        product = (a.bfloat16() @ a.bfloat16()).float()
        conv = torch.nn.functional.conv2d(image.bfloat16(), kernel.bfloat16()).float()
        soft = torch.softmax(a.bfloat16() * 30, dim=-1).float()
        close = torch.allclose(product, a @ a, rtol=0.05, atol=2.0)
        return bool(torch.isfinite(product).all() and torch.isfinite(conv).all()
                    and torch.isfinite(soft).all() and close)
    except RuntimeError as error:
        print("bf16 test failed:", error)
        return False


# Never fp16: Z-Image overflows it. bf16 keeps fp32's range (native from compute 8.0, emulated
# on the T4/P100: slower but correct); fp32 if bf16 does not work at all.
DTYPE = "bf16" if MAJOR >= 8 or bf16_works() else "fp32"
print(f"GPU: {torch.cuda.get_device_name(0)} (compute {MAJOR}.{MINOR}), "
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB, training dtype {DTYPE}"
      + ("" if MAJOR >= 8 else " (emulated on this GPU: slower)"))

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SCRATCH, exist_ok=True)
os.environ["HF_HOME"] = f"{SCRATCH}/hf"  # base models stay out of /kaggle/working
STATUS: dict[str, object] = {
    "status": "started", "dtype": DTYPE, "gpu": torch.cuda.get_device_name(0),
}


def save_status(**changes: object) -> None:
    STATUS.update(changes)
    Path(OUTPUT_DIR, "status.json").write_text(json.dumps(STATUS, indent=2))


save_status()

In [ ]:
# The dataset from `ca lora prepare`, found wherever Kaggle put it (unpacked or still zipped,
# any folder depth, any image-extension case). A missing or untidy caption is fixed: every
# caption starts with the trigger word, which is the most common first word of the captions.
import shutil
import zipfile
from collections import Counter

IMAGE_TYPES = {".jpg", ".jpeg", ".png", ".webp"}


def images_in(folder: Path) -> list[Path]:
    return sorted(path for path in folder.iterdir()
                  if path.is_file() and path.suffix.lower() in IMAGE_TYPES)


def caption_for(image: Path) -> str | None:
    for path in image.parent.iterdir():
        if path.suffix.lower() == ".txt" and path.stem.lower() == image.stem.lower():
            return path.read_text(encoding="utf-8-sig", errors="ignore").strip() or None
    return None


for packed in Path("/kaggle/input").rglob("*.zip"):
    target = Path(SCRATCH, "unzipped", packed.stem)
    if not target.exists():
        try:
            with zipfile.ZipFile(packed) as bundle:
                bundle.extractall(target)
        except zipfile.BadZipFile:
            print("skipped (not a zip):", packed)
roots = [Path("/kaggle/input"), Path(SCRATCH, "unzipped")]
folders = [folder for root in roots if root.exists() for folder in [root, *root.rglob("*")]
           if folder.is_dir() and "lora_output" not in folder.parts]
found = max(folders, key=lambda folder: sum(caption_for(image) is not None
                                             for image in images_in(folder)), default=None)
images = images_in(found) if found is not None else []
captions = {image: caption_for(image) for image in images}
firsts = Counter(text.split(",")[0].strip() for text in captions.values() if text)
if len(images) < MIN_IMAGES or not firsts:
    save_status(status=f"failed: no dataset with at least {MIN_IMAGES} captioned images")
    raise SystemExit("Attach the private dataset made from `ca lora prepare` (Add Input).")
TRIGGER = firsts.most_common(1)[0][0]
DATASET = Path(SCRATCH, "dataset")
shutil.rmtree(DATASET, ignore_errors=True)
DATASET.mkdir(parents=True)
fixed = []
for number, image in enumerate(images, start=1):
    text = captions[image] or f"{TRIGGER}, a man"
    if not text.lower().startswith(TRIGGER.lower()):
        text = f"{TRIGGER}, {text}"
    if text != captions[image]:
        fixed.append(image.name)
    stem = f"{number:02d}"
    shutil.copy(image, DATASET / f"{stem}{image.suffix.lower()}")
    Path(DATASET, f"{stem}.txt").write_text(text + "\n", encoding="utf-8")
JOB = f"{TRIGGER}_z_image_turbo"
print(f"Dataset: {found} -> {len(images)} images, trigger {TRIGGER!r}, job {JOB}")
if fixed:
    print("captions added or fixed for:", ", ".join(fixed))
save_status(dataset=str(found), images=len(images), trigger=TRIGGER, job=JOB,
            captions_fixed=fixed)

In [ ]:
# AI Toolkit (MIT): clone, then install its requirements one by one. "-r" includes are
# followed, torch/torchvision/torchaudio are skipped (Kaggle's CUDA build stays), and a pin
# that fails is retried without its version.
import re
import sys
from importlib.metadata import PackageNotFoundError, version

TOOLKIT = Path(SCRATCH, "ai-toolkit")
if not (TOOLKIT / "run.py").exists():
    sh(f"git clone --depth 1 {TOOLKIT_REPO} {TOOLKIT}")
commit = subprocess.run(["git", "-C", str(TOOLKIT), "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("AI Toolkit commit:", commit)


def requirement_lines(path: Path, seen: set[Path]) -> list[str]:
    if path in seen or not path.is_file():
        return []
    seen.add(path)
    lines: list[str] = []
    for raw in path.read_text().splitlines():
        line = raw.split(" #", 1)[0].strip()
        if not line or line.startswith("#"):
            continue
        include = re.match(r"^(?:-r|--requirement)\s*=?\s*(\S+)", line)
        if include:
            lines += requirement_lines((path.parent / include.group(1)).resolve(), seen)
        elif not line.startswith("-"):
            lines.append(line)
    return lines


def pip(spec: str) -> bool:
    command = [sys.executable, "-m", "pip", "install", "-q", spec]
    return subprocess.run(command, capture_output=True, text=True).returncode == 0


relaxed, failed = [], []
for spec in requirement_lines(TOOLKIT / "requirements.txt", set()):
    plain = re.match(r"^([A-Za-z0-9][A-Za-z0-9._-]*)(\[[^\]]*\])?", spec)
    name = plain.group(1).lower().replace("_", "-") if plain else spec
    if name in SKIP_PACKAGES:
        continue
    spec = PINNED.get(name, spec)
    if pip(spec):
        continue
    loose = f"{plain.group(1)}{plain.group(2) or ''}" if plain else spec
    if loose != spec and pip(loose):
        relaxed.append(f"{spec} -> {loose}")
    else:
        failed.append(spec)
print("relaxed pins:", relaxed or "none")
print("could not install:", failed or "none")
def installed(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "-"


VERSIONS = {name: installed(name) for name in (
    "torch", "optimum-quanto", "torchao", "bitsandbytes", "diffusers", "transformers",
    "accelerate", "safetensors", "peft")}
VERSIONS["python"] = sys.version.split()[0]
print("versions:", VERSIONS)
save_status(toolkit_commit=commit, relaxed_pins=relaxed, install_failed=failed,
            versions=VERSIONS)

In [ ]:
# The Z-Image arch name as this toolkit version registers it, and the newest Turbo adapter.
from huggingface_hub import list_repo_files

ARCH_PATTERN = re.compile(r"""arch\s*=\s*['"]([A-Za-z0-9_:.-]*z_?image[A-Za-z0-9_:.-]*)['"]""",
                          re.IGNORECASE)
arches = sorted({
    match.group(1)
    for path in TOOLKIT.rglob("*.py")
    for match in ARCH_PATTERN.finditer(path.read_text(errors="ignore"))
})
print("Z-Image arches in this AI Toolkit:", arches)
if not arches:
    save_status(status="failed: this AI Toolkit version has no Z-Image arch")
    raise SystemExit("No Z-Image support found in AI Toolkit (see status.json).")
ARCH = "zimage" if "zimage" in arches else arches[0]
adapters = sorted(name for name in list_repo_files(ADAPTER_REPO) if name.endswith(".safetensors"))
if not adapters:
    save_status(status=f"failed: no adapter file in {ADAPTER_REPO}")
    raise SystemExit(f"No training adapter found in {ADAPTER_REPO}.")
ADAPTER = f"{ADAPTER_REPO}/{adapters[-1]}"  # e.g. ..._v2 over ..._v1
print("Arch:", ARCH, "| adapter:", ADAPTER)
save_status(arch=ARCH, adapter=ADAPTER)

In [ ]:
# Resume: the newest checkpoint in /kaggle/working or in an attached earlier output, if it
# learned something (finite weights, LoRA up/B weights not all zero). AI Toolkit continues
# from the newest checkpoint it finds in the job folder.
from safetensors.torch import load_file

JOB_DIR = Path(OUTPUT_DIR, JOB)
STEP_PATTERN = re.compile(r"_(\d+)\.safetensors$")


def step_of(path: Path) -> int:
    match = STEP_PATTERN.search(path.name)
    return int(match.group(1)) if match else 10**9  # the final file: after every step


def learned(path: Path) -> bool:
    state = load_file(str(path))
    if not all(bool(torch.isfinite(tensor.float()).all()) for tensor in state.values()):
        return False
    ups = [tensor for key, tensor in state.items() if "lora_B" in key or "lora_up" in key]
    return not ups or any(bool(tensor.abs().max() > 0) for tensor in ups)


sources = [JOB_DIR] + [folder for folder in Path("/kaggle/input").rglob(JOB) if folder.is_dir()]
saved = [path for folder in sources if folder.exists() for path in folder.glob("*.safetensors")]
RESUMED_FROM = None
for newest in sorted(saved, key=step_of, reverse=True):
    if learned(newest):
        RESUMED_FROM = newest
        break
    print("not usable (learned nothing):", newest)
if RESUMED_FROM is None and JOB_DIR.exists() and any(JOB_DIR.glob("*.safetensors")):
    aside = JOB_DIR.with_name(f"{JOB}_unusable_{int(time.time())}")
    JOB_DIR.rename(aside)  # kept for reference; AI Toolkit starts fresh
    print("set aside:", aside)
if RESUMED_FROM is not None and RESUMED_FROM.parent != JOB_DIR:
    JOB_DIR.mkdir(parents=True, exist_ok=True)
    for path in RESUMED_FROM.parent.iterdir():  # checkpoints, optimizer state, samples
        target = JOB_DIR / path.name
        if path.is_dir():
            shutil.copytree(path, target, dirs_exist_ok=True)
        elif step_of(path) <= step_of(RESUMED_FROM) or not path.name.endswith(".safetensors"):
            shutil.copy(path, target)
print("Resume from:", RESUMED_FROM or "nothing usable: a fresh start")
save_status(resumed_from=str(RESUMED_FROM) if RESUMED_FROM else None)

In [ ]:
# The AI Toolkit config (YAML; JSON strings and numbers are valid YAML scalars). Everything
# AI Toolkit saves goes to /kaggle/working/lora_output/<job>/.
CONFIG = Path(OUTPUT_DIR, f"{JOB}.yaml")


def q(value: object) -> str:
    return json.dumps(value)


def write_config(resolutions: list[int]) -> None:
    prompts = "\n".join(f"          - {q(prompt)}" for prompt in SAMPLE_PROMPTS)
    CONFIG.write_text(f"""job: extension
config:
  name: {q(JOB)}
  process:
    - type: "sd_trainer"
      training_folder: {q(OUTPUT_DIR)}
      device: "cuda:0"
      trigger_word: {q(TRIGGER)}
      network:
        type: "lora"
        linear: {RANK}
        linear_alpha: {RANK}
      save:
        dtype: "float16"
        save_every: {SAVE_EVERY}
        max_step_saves_to_keep: {KEEP_CHECKPOINTS}
      datasets:
        - folder_path: {q(str(DATASET))}
          caption_ext: "txt"
          caption_dropout_rate: 0.05
          cache_latents_to_disk: true
          resolution: {q(resolutions)}
      train:
        batch_size: 1
        cache_text_embeddings: true
        skip_first_sample: true
        steps: {STEPS}
        gradient_accumulation: 1
        train_unet: true
        train_text_encoder: false
        gradient_checkpointing: true
        noise_scheduler: "flowmatch"
        optimizer: {q(OPTIMIZER)}
        lr: {LEARNING_RATE}
        dtype: {q(DTYPE)}
      model:
        name_or_path: {q(BASE_MODEL)}
        arch: {q(ARCH)}
        assistant_lora_path: {q(ADAPTER)}
        quantize: true
        qtype: "qfloat8"
        quantize_te: true
        qtype_te: "qfloat8"
        low_vram: true
      sample:
        sampler: "flowmatch"
        sample_every: {SAVE_EVERY}
        width: {SAMPLE_SIZE}
        height: {SAMPLE_SIZE}
        prompts:
{prompts}
        neg: ""
        seed: 42
        walk_seed: true
        guidance_scale: 1
        sample_steps: 8
meta:
  name: "[name]"
  version: "1.0"
""")


write_config(RESOLUTIONS)
print(CONFIG.read_text())

In [ ]:
# Train. Never raises: whatever happens, the next cell packs every finished checkpoint.
# The guard stops the run when the loss is 0, NaN or infinite for the first GUARD_STEPS steps
# (AI Toolkit prints "loss is nan" and reports a non-finite loss as 0).
import signal

LOG = Path(OUTPUT_DIR, "train_log.txt")
DEADLINE = time.time() + TIME_BUDGET_HOURS * 3600
PROGRESS = re.compile(r"(\d+)/(\d+) \[[^\]]*?loss:\s*([-+0-9.eE]+|nan|inf)", re.IGNORECASE)


def checkpoints() -> list[Path]:
    return sorted(JOB_DIR.glob("*.safetensors"), key=step_of)


def loss_problem(text: str) -> str | None:
    """None while the loss looks real; a reason once the first GUARD_STEPS steps are bad."""
    seen = {int(step): value.lower() for step, _, value in PROGRESS.findall(text)}
    if len(seen) < 2 or max(seen) - min(seen) < GUARD_STEPS:
        return None
    first = [value for step, value in sorted(seen.items()) if step < min(seen) + GUARD_STEPS]
    bad = [value for value in first if value in ("nan", "inf", "-inf") or float(value) == 0.0]
    if len(bad) == len(first):
        nans = text.count("loss is nan")
        return (f"loss was 0/NaN for the first {GUARD_STEPS} steps ({nans} 'loss is nan' lines; "
                f"dtype {DTYPE}): nothing would be learned")
    return None


def memory_line(pid: int, step: int) -> str:
    """Free RAM, the trainer's RAM and GPU 0 memory: the next crash is easy to place."""
    info = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
    avail = int(info["MemAvailable"].split()[0]) / 2**20
    total = int(info["MemTotal"].split()[0]) / 2**20
    rss = 0.0
    status = Path(f"/proc/{pid}/status")
    if status.exists():
        for line in status.read_text().splitlines():
            if line.startswith("VmRSS:"):
                rss = int(line.split()[1]) / 2**20
    gpu = subprocess.run(
        ["nvidia-smi", "--id=0", "--query-gpu=memory.used,memory.total",
         "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    return (f"[memory] step {step}: RAM available {avail:.1f} of {total:.1f} GB, trainer "
            f"{rss:.1f} GB; GPU 0 {gpu or 'n/a'}")


def stop(process: subprocess.Popen[bytes], wait_s: int) -> None:
    os.killpg(process.pid, signal.SIGINT)  # AI Toolkit stops at Ctrl+C
    try:
        process.wait(timeout=wait_s)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)


def run_training(resolutions: list[int]) -> str:
    write_config(resolutions)
    started = time.time()
    with LOG.open("a") as log:
        log.write(f"\n=== training at {resolutions}, dtype {DTYPE} ===\n")
        log.flush()
        offset = LOG.stat().st_size  # the guard reads this run's part only
        log.write(f"versions: {VERSIONS}\n")
        log.flush()
        # One GPU; a segfault prints the Python stack of every thread into this log.
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "PYTHONFAULTHANDLER": "1"}
        process = subprocess.Popen([sys.executable, "run.py", str(CONFIG)], cwd=TOOLKIT,
                                   stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
                                   env=env)
        shown = started
        logged = -1  # the last step whose memory line was written
        while process.poll() is None:
            time.sleep(30)
            with LOG.open("rb") as reader:
                reader.seek(offset)
                text = reader.read().decode(errors="ignore")
            steps = [int(step) for step, _, _ in PROGRESS.findall(text)]
            if steps and steps[-1] // MEMORY_LOG_EVERY > logged // MEMORY_LOG_EVERY:
                logged = steps[-1]
                log.write(f"\n{memory_line(process.pid, logged)}\n")
                log.flush()
            problem = loss_problem(text)
            if problem:
                stop(process, 60)
                return f"failed: {problem}"
            if time.time() > DEADLINE:
                stop(process, 300)
                return "stopped: time budget reached"
            if time.time() - shown > 600:  # progress every 10 minutes
                shown = time.time()
                last = PROGRESS.findall(text)[-1:]
                step = "no step yet"
                if last:
                    step = f"step {last[0][0]}/{last[0][1]}, loss {last[0][2]}"
                print(f"[{(shown - started) / 3600:.1f} h] {step}; "
                      f"{len(checkpoints())} checkpoints", flush=True)
    if process.returncode == 0:
        return "done"
    text = LOG.read_text(errors="ignore")
    if "out of memory" in text.lower() and not checkpoints():
        return "oom"
    return f"failed: exit code {process.returncode} (see train_log.txt)"


try:
    result = run_training(RESOLUTIONS)
    if result == "oom":
        print("Out of GPU memory before the first checkpoint: trying once at", FALLBACK_RESOLUTIONS)
        STATUS["notes"] = ["out of memory at 1024: trained at 512/768"]
        result = run_training(FALLBACK_RESOLUTIONS)
        if result == "oom":
            result = "failed: out of GPU memory also at 512/768 (see train_log.txt)"
except Exception as error:  # noqa: BLE001 - the outputs are packed whatever happened
    result = f"failed: {type(error).__name__}: {error}"
save_status(status=result, checkpoints=[path.name for path in checkpoints()])
print("Training:", result)

In [ ]:
# Pack every checkpoint, sample image, the config, the log and status.json into one zip.
ARCHIVE = Path(WORK, f"{JOB}_outputs.zip")
with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(Path(OUTPUT_DIR).rglob("*")):
        if path.is_file() and "_latent_cache" not in path.parts:
            bundle.write(path, path.relative_to(OUTPUT_DIR))
samples = sorted(Path(JOB_DIR, "samples").glob("*"))
print(f"Status: {STATUS['status']}")
print(f"Checkpoints ({len(checkpoints())}):", *[path.name for path in checkpoints()], sep="\n  ")
print(f"Sample images: {len(samples)} in {Path(JOB_DIR, 'samples')}")
print(f"Everything in one file: {ARCHIVE} ({ARCHIVE.stat().st_size / 2**20:.0f} MB)")
print("Next: pick a checkpoint from its samples, then run kaggle_test_lora.ipynb with it.")